# tsconfig 与项目组织

学习目标：能组织明确的编译项目，区分源码范围、输出与类型环境，并使别名、包入口及检查配置匹配实际宿主。

前置知识：tsc -p、JSON 配置、ESM、package.json、模块解析、声明文件、类型检查与运行时的区别。

适用版本：TypeScript 7.0.2、Node.js 24.11.0；ES 模块，开启 strict；另启用 noUncheckedIndexedAccess、exactOptionalPropertyTypes、noImplicitOverride、isolatedModules、sourceMap。

环境准备：[环境配置与运行](README.md)。

工作目录：content/编程语言/typescript。

配套脚本：位于 scripts/20-project-config/。

1. [tsconfig.base.json](scripts/20-project-config/tsconfig.base.json)：配套实现与示例。
2. [tsconfig.json](scripts/20-project-config/tsconfig.json)：配套实现与示例。
3. [included/excluded.ts](scripts/20-project-config/included/excluded.ts)：配套实现与示例。
4. [included/extra.ts](scripts/20-project-config/included/extra.ts)：配套实现与示例。
5. [value.ts](scripts/20-project-config/value.ts)：配套实现与示例。
6. [tsconfig.typeroots.json](scripts/20-project-config/tsconfig.typeroots.json)：配套实现与示例。
7. [main.ts](scripts/20-project-config/main.ts)：配套实现与示例。
8. [tsconfig.emit-errors.json](scripts/20-project-config/tsconfig.emit-errors.json)：配套实现与示例。
9. [alias-only.ts](scripts/20-project-config/alias-only.ts)：配套实现与示例。
10. [package.json](scripts/20-project-config/package.json)：配套实现与示例。
11. [prepare-output.mjs](scripts/20-project-config/prepare-output.mjs)：配套实现与示例。
12. [sourcemap-error.ts](scripts/20-project-config/sourcemap-error.ts)：配套实现与示例。
13. [unused.ts](scripts/20-project-config/unused.ts)：配套实现与示例。
14. [tsconfig.unused.json](scripts/20-project-config/tsconfig.unused.json)：配套实现与示例。
15. [tsconfig.json](scripts/20-project-config/tsconfig.json)：本章独立项目配置。
16. [type-errors.ts](scripts/20-project-config/type-errors.ts)、[tsconfig.errors.json](scripts/20-project-config/tsconfig.errors.json)：单独检查的类型反例。

Step 1：检查本章正常示例的类型。

```bash
npm run check:20
```

Step 2：生成本章 JavaScript。

```bash
npm run build:20
```

Step 3：运行本章正常示例。

```bash
npm run run:20
# 正常退出；各段预期输出见代码注释。
```

同一文件的片段按正文顺序衔接，前文定义在后续片段中继续使用。直接运行完整项目；反例使用独立配置，不进入正常运行入口。

## 1 compilerOptions 与配置继承

tsconfig.json 定义一个检查和生成项目；compilerOptions 保存编译选项，files、include 等顶层字段控制初始源码范围。tsc -p 明确指定要用哪个项目，避免依赖当前目录的意外配置。

extends 先加载基础配置，派生配置再覆盖；相对路径按最初声明该路径的配置文件解析。files、include、exclude 是覆盖而非自动拼接。这里基础配置只开启 switch 分支贯穿检查，章节配置明确写出环境和输出条件。

对应 [tsconfig.base.json](scripts/20-project-config/tsconfig.base.json)。

```json
{
  "compilerOptions": {
    "noFallthroughCasesInSwitch": true
  }
}
```

## 2 本章完整项目配置

rootDir 与 outDir 都相对本配置文件。这里 rootDir 为当前章节，输出到技术目录内 .build/20-project-config；不会把别章的配置或全局声明一起纳入。

strict 的附加检查、paths 和其他选项在下面逐项解释。编译器选项属于本章教学目标，因此完整配置集中展示一次。

对应 [tsconfig.json](scripts/20-project-config/tsconfig.json)。

```json
{
  "compilerOptions": {
    "strict": true,
    "target": "ES2025",
    "module": "NodeNext",
    "moduleResolution": "NodeNext",
    "lib": [
      "ES2025"
    ],
    "types": [
      "node"
    ],
    "rootDir": ".",
    "outDir": "../../.build/20-project-config",
    "noEmitOnError": true,
    "noUncheckedIndexedAccess": true,
    "exactOptionalPropertyTypes": true,
    "noImplicitOverride": true,
    "isolatedModules": true,
    "sourceMap": true,
    "skipLibCheck": false,
    "paths": {
      "@lesson/value": [
        "./value.ts"
      ]
    }
  },
  "files": [
    "main.ts",
    "alias-only.ts",
    "sourcemap-error.ts"
  ],
  "extends": "./tsconfig.base.json",
  "include": [
    "included/**/*.ts"
  ],
  "exclude": [
    "included/excluded.ts"
  ]
}
```

## 3 files、include、exclude 与导入关系

分开看“搜索初始输入”和“沿导入加入依赖”，就不会把 exclude 误当成禁止访问文件。

files 列出明确的入口文件，缺失会报错；include 按相对配置文件的模式搜索，双星号目录模式可以匹配子目录；exclude 仅过滤 include 找到的文件，不能阻止显式导入或 files 把文件重新纳入。

本例 included/excluded.ts 没有通过 include 的筛选，但 main.ts 的导入把它重新纳入。下面分别画这两条路径；观察项目范围时要跟到依赖终点，不能把 exclude 当访问隔离。

![exclude 只过滤 include 的搜索结果。主入口的显式导入仍会把被排除文件带入检查项目。](image/illustration/20-01-project-input-inclusion.svg)

图示说明（依据篇末官方文档自绘）：上排概括 include 与 exclude 的关系；下排只画本例 main.ts 的一条导入边。

运行下面 --listFiles 命令，定位 included/excluded.ts，再回看 main.ts 的导入来解释它为何仍在列表里。

Step 1：查看本章检查实际纳入的文件。

```bash
npm run check:20 -- --listFiles
# 文件列表仍包含 scripts/20-project-config/included/excluded.ts。
```

对应 [included/excluded.ts](scripts/20-project-config/included/excluded.ts)。

```typescript
export const importedDespiteExclude = 2;
```

## 4 rootDir、outDir 与目录结构

rootDir 决定源文件在输出目录中的相对层级，并不选择输入文件；需要输出的源码必须处于它覆盖的范围。outDir 指定产物位置，也不会自动清除以前的旧文件。

下面 included/extra.ts 由 include 纳入，即使主入口没有导入它，也会检查并输出为 .build/20-project-config/included/extra.js。TypeScript 6/7 的配置项目 rootDir 默认值已改为配置所在目录，本章显式设置，避免沿用旧版“推断公共源码目录”的假设。

对应 [included/extra.ts](scripts/20-project-config/included/extra.ts)。

```typescript
export const extra = "由 include 纳入";
```

## 5 target、lib 与宿主支持

target 控制语法输出的目标级别，lib 提供标准 API 的声明集合；二者都不安装 polyfill。module/moduleResolution NodeNext 表达 Node 模块规则，不能替代实际 Node 版本条件。

本章使用 ES2025 声明和 Node.js 24.11.0，只调用已在该宿主检查过的简单能力。若加入 DOM 声明，也不会让 Node 自动出现 document；API 的声明、实现和可用时机需要分别确认。

对应 [value.ts](scripts/20-project-config/value.ts)。

```typescript
export const amount = 7;
```

## 6 types 与 typeRoots

types 列出自动纳入的类型包名称，typeRoots 指定类型包所在的目录；两者不是普通 .d.ts 文件清单。本章默认从现有 node_modules 查找 node，不把其他类型包的全局声明意外加入。

TypeScript 6/7 的 types 默认是空数组，所以不能沿用旧版“自动纳入所有 @types”的默认假设。types 控制自动纳入的类型包；模块导入仍按 moduleResolution 查找声明。自 TypeScript 5.1 起，常规策略无法解析模块时，还会尝试相对于显式 typeRoots 查找类型包，因此 typeRoots 也可能影响导入的类型解析结果。下面的独立配置演示显式类型包目录，路径相对当前章节。

Step 1：检查显式 typeRoots 的配置。

```bash
npm run check:20:typeroots
# 预期正常退出，仍能使用 Node 类型。
```

对应 [tsconfig.typeroots.json](scripts/20-project-config/tsconfig.typeroots.json)。

```json
{
  "extends": "./tsconfig.json",
  "compilerOptions": { "typeRoots": ["../../node_modules/@types"], "noEmit": true },
  "files": ["main.ts"],
  "include": []
}
```

## 7 strict 与独立附加检查

strict 汇总一组严格检查，包括隐式 any、空值、函数类型和类字段初始化等；未来编译器版本可能扩大这组检查。下面三项额外显式开启，不把它们误认为 strict 自动包含。

| 完整名称 | 中文名称／含义 | 本章观察 |
| --- | --- | --- |
| noUncheckedIndexedAccess | 索引访问包含未找到的可能 | 数组读取后处理 undefined |
| exactOptionalPropertyTypes | 精确可选属性 | 缺省属性不同于显式 undefined |
| noImplicitOverride | 显式覆写声明 | 子类覆写使用 override |

这些选项不会替运行时过滤输入，只让相关边界在代码中更明确。

对应 [main.ts](scripts/20-project-config/main.ts)。

```typescript
import { amount } from "#value";
import { amount as publicAmount } from "ts-b-config-demo/value";
import { importedDespiteExclude } from "./included/excluded.js";
const scores: number[] = [amount];
const first = scores[0];
const safeScore = first === undefined ? 0 : first;
type Options = { label?: string };
const options: Options = {};
class Base { title() { return "基础"; } }
class Derived extends Base { override title() { return "项目"; } }
console.log(safeScore, publicAmount, importedDespiteExclude, options.label, new Derived().title());
// 预期输出：7 7 2 undefined 项目
```

## 8 isolatedModules、生成控制与声明检查

isolatedModules 检查单文件转换工具难以正确处理的写法，并不表示编译器只检查一个文件，也不生成依赖的实现。类型导出写 export type 等明确形式有助于消除转换歧义。

noEmit 无条件禁止本次生成，适合只检查类型；noEmitOnError 在发现诊断时阻止本次输出，不会删除以前成功构建的旧产物。不要在失败后运行旧文件并误认为新代码有效。

skipLibCheck 为 true 会跳过声明文件内部的完整类型检查，可能掩盖不一致的依赖声明。本章保留 false；解决冲突应先查版本与重复依赖，不能只靠跳过检查。

Step 1：在独立输出目录演示错误阻止生成。

```bash
npm run errors:20:emit
# 预期退出码 1，不生成 .build/20-project-config/errors/type-errors.js。
```

对应 [tsconfig.emit-errors.json](scripts/20-project-config/tsconfig.emit-errors.json)。

```json
{
  "extends": "./tsconfig.json",
  "compilerOptions": { "outDir": "../../.build/20-project-config/errors", "noEmit": false },
  "files": ["type-errors.ts"],
  "include": []
}
```

## 9 paths 不改写运行时说明符

检查器把别名找到源码之后，运行时是否也有对应入口，需要另走一遍解析路径。

本章 paths 将 @lesson/value 映射到 ./value.ts，让编译器能查到类型；输出中的导入仍是 @lesson/value。Node 不读取 tsconfig 的 paths，因此下面代码虽然通过检查，单独运行会找不到包。

TypeScript 7 已移除 baseUrl；paths 的目标使用相对配置的完整路径即可。不要把移除选项的错误当成普通源码类型错误。

![paths 解析成功不等于 Node 能加载。本例仅给 TypeScript 配置了 @lesson/value 的映射。](image/illustration/20-02-paths-runtime-boundary.svg)

图示说明（依据篇末官方文档自绘）：图对应本例直接使用 tsc 生成并交给 Node 的条件，不讨论另有工具主动改写路径的项目。

运行反例前先比较 alias-only.ts 与生成的 JavaScript：两处说明符是否仍相同，再阅读 Node 的缺包诊断。

Step 1：在完成 build:20 后运行只有类型别名映射的反例。

```bash
node .build/20-project-config/alias-only.js
# 预期退出码 1，诊断包含 ERR_MODULE_NOT_FOUND 和 @lesson/value。
```

对应 [alias-only.ts](scripts/20-project-config/alias-only.ts)。

```typescript
import { amount } from "@lesson/value";
console.log(amount); // tsc 保留这个说明符；Node 没有同名包。
```

## 10 用包 imports 与 exports 对齐宿主

package.json 的 imports 定义包内部以 # 开头的说明符；exports 定义按包名可访问的入口。NodeNext 理解这些映射，Node 也按相同包边界加载实现；主入口已经分别使用 #value 和包的自引用子路径。

exports 的相对目标以 ./ 开头，不能指向包外。这个 package.json 仅给本章建立包边界，没有新增依赖。

对应 [package.json](scripts/20-project-config/package.json)。

```json
{
  "name": "ts-b-config-demo",
  "private": true,
  "type": "module",
  "imports": { "#value": "./value.js" },
  "exports": { "./value": "./value.js" }
}
```

## 11 把包配置带到输出目录

tsc 的 TypeScript 输出不等于完整部署目录。这里还需要把 package.json 放到生成文件旁，保证运行时包边界和源码一致；prepare-output.mjs 只复制这份已有配置，不改写模块说明符。

build:20 在 tsc 成功后调用下面的辅助文件。若直接使用 tsc -p，运行前也必须单独执行该复制步骤。路径相对于本辅助模块，而非调用终端的目录。

对应 [prepare-output.mjs](scripts/20-project-config/prepare-output.mjs)。

```javascript
import { copyFileSync, mkdirSync } from "node:fs";
const output = new URL("../../.build/20-project-config/", import.meta.url);
mkdirSync(output, { recursive: true });
copyFileSync(new URL("./package.json", import.meta.url), new URL("./package.json", output));
```

## 12 sourceMap 与异常定位

sourceMap 生成 .js.map，将输出代码位置映射回源码；它不改变业务逻辑，也不会自动启动调试器。Node 的 --enable-source-maps 可以让异常堆栈尽力显示源码位置。

下面的异常独立运行，正常入口不执行它。产物和源映射需来自同一次构建，不能拿旧 map 对照新文件。

Step 1：运行带源映射的定位反例。

```bash
node --enable-source-maps .build/20-project-config/sourcemap-error.js
# 预期退出码 1，包含 source-map-demo，堆栈指向 sourcemap-error.ts。
```

对应 [sourcemap-error.ts](scripts/20-project-config/sourcemap-error.ts)。

```typescript
export {};
throw new Error("source-map-demo");
```

## 13 类型检查与代码规范检查的分工

noUnusedLocals 与 noUnusedParameters 可以让编译器报告未使用的局部变量和参数；它们不负责所有格式、命名和团队规则。完整代码规范通常由编辑器格式化和独立规范工具承担，不能把 tsc 通过写成“所有规范检查通过”。

unused.ts 用两个未使用名称展示本章额外的使用检查，配置只启用这两项检查并禁止生成；该文件不进入正常项目。

Step 1：运行未使用名称的反例。

```bash
npm run errors:20:unused
# 预期退出码 1，TS6133 指向 unusedLabel 与 unusedValue。
```

对应 [unused.ts](scripts/20-project-config/unused.ts)。

```typescript
export function identity(value: number, unusedLabel: string): number { return value; }
const unusedValue = 1;
```

## 14 未使用名称检查的独立配置

在基础项目上开启额外检查时，明确源码范围和输出策略。下面覆盖 files 与 include，避免把其他教学例子的未使用类型别名一起检查。

对应 [tsconfig.unused.json](scripts/20-project-config/tsconfig.unused.json)。

```json
{
  "extends": "./tsconfig.json",
  "compilerOptions": { "noUnusedLocals": true, "noUnusedParameters": true, "noEmit": true },
  "files": ["unused.ts"],
  "include": []
}
```

## 15 固定版本与编辑器一致性

课程 package.json 与锁文件固定 TypeScript 7.0.2、@types/node 和运行工具条件；已有依赖按锁文件准备，不在章节里临时升级。npm run 使用本项目的命令入口，避免终端意外调用另一套全局编译器。

编辑器必须采用兼容 TypeScript 7 的语言服务，并打开正确的配置项目。编辑器提示与命令行不一致时，先核对版本、文件所属项目及配置，不通过关闭 strict 掩盖差异。TypeScript 7 原生实现不意味着旧工具的编译器 API 插件都兼容。

迁移旧项目还要核查移除项：ES5 目标、node10/classic 解析、baseUrl 等不能直接搬到本基线。保持宿主、输出、类型环境和工具版本同步，比单独提高 target 更能说明运行条件。

## 16 检查类型边界

下面的 [type-errors.ts](scripts/20-project-config/type-errors.ts) 只用于检查，不执行。逐项阅读注释，修正时保留原本需求，不通过断言或关闭检查掩盖错误。

```typescript
const values: number[] = [1];
const first: number = values[0]; // 开启索引检查后可能为 undefined。
const options: { label?: string } = { label: undefined }; // 可选不等于允许显式 undefined。
class Base { title() { return "基础"; } }
class Child extends Base { title() { return "子类"; } } // 需要 override。
export {};
// 预期诊断包含：TS2322, TS2375, TS4114。
```

Step 1：单独检查反例并对照错误位置与原因。

```bash
npm run errors:20
# 本章固定编译器预期退出码为 1；正常项目命令的退出码为 0。
```

## 本章小结

配置决定哪些文件被检查、怎样输出及采用哪些声明。exclude 不隔离导入，paths 不改写输出；Node 包边界也需要出现在产物中。固定版本并分别检查类型、生成、运行和代码规范。

## 练习

1. 删除 main.ts 对 excluded.js 的导入及 console.log 对 importedDespiteExclude 的使用，再查看文件列表；确认 excluded.ts 消失，而 included/extra.ts 仍存在。

2. 把 values[0] 的读取改为先检查 undefined，再运行错误配置，确认对应诊断消失。

3. 删除输出目录中的 package.json 后运行 main.js，观察包映射失败；执行 prepare-output.mjs 恢复后核对输出恢复。

4. 修改 source-map-demo 的异常行位置，重新构建后运行带映射的命令，核对堆栈跟随新源码位置。
5. 分别在两张图上指出 exclude 和 paths 影响的是哪条路径；核对更改它们是否会自动删除显式导入或重写输出说明符，不能仅凭检查通过下结论。

### 提示

1. 删除导入时也删除 console.log 中 importedDespiteExclude 的使用。
2. 在 `first` 赋值前缩窄读取结果的类型；其他两项错误保留。
3. 使用本章已有生成目录，不改课程根 package.json。
4. 修改 .ts 源文件并重新构建，不直接编辑 .js.map。
5. 分别沿图中的初始输入路径与运行路径观察。

### 参考解析

1. 移除导入及其使用后，excluded.ts 不再由依赖关系加入；extra.ts 仍由 include 纳入。完成后恢复原示例。
2. `const value = values[0]` 后，在 `value !== undefined` 分支内把它赋给 number。这里判断专用于索引访问缩窄教学；可选属性和 override 的两处诊断仍应存在。
3. 删除生成目录的包配置使 #value 等映射不再属于原来的包边界，Node 加载失败；运行 prepare-output.mjs 后恢复。
4. 重新构建产生配套的 .js 与 .js.map，启用映射时堆栈中的 .ts 行号应跟随异常行变化。
5. exclude 只过滤 include 搜索结果，显式导入仍加入依赖；paths 只参与检查器解析，生成的说明符仍保持原文。

## 参考与引用来源

| 来源 | 支持的知识点与定位 |
| --- | --- |
| TypeScript 官方文档 | [TSConfig：顶层 files、include、exclude、compilerOptions；exclude 的导入例外](https://www.typescriptlang.org/tsconfig/#exclude)；[extends](https://www.typescriptlang.org/tsconfig/extends.html)；[rootDir](https://www.typescriptlang.org/tsconfig/rootDir.html)；[outDir](https://www.typescriptlang.org/tsconfig/outDir.html)；[target](https://www.typescriptlang.org/tsconfig/target.html)；[lib](https://www.typescriptlang.org/tsconfig/lib.html)；[types](https://www.typescriptlang.org/tsconfig/types.html)；[typeRoots](https://www.typescriptlang.org/tsconfig/typeRoots.html)；[5.1：typeRoots Are Consulted In Module Resolution](https://www.typescriptlang.org/docs/handbook/release-notes/typescript-5-1.html#typeroots-are-consulted-in-module-resolution)；[strict](https://www.typescriptlang.org/tsconfig/strict.html)；[noUncheckedIndexedAccess](https://www.typescriptlang.org/tsconfig/noUncheckedIndexedAccess.html)；[exactOptionalPropertyTypes](https://www.typescriptlang.org/tsconfig/exactOptionalPropertyTypes.html)；[noImplicitOverride](https://www.typescriptlang.org/tsconfig/noImplicitOverride.html)；[isolatedModules](https://www.typescriptlang.org/tsconfig/isolatedModules.html)；[noEmit](https://www.typescriptlang.org/tsconfig/noEmit.html)；[noEmitOnError](https://www.typescriptlang.org/tsconfig/noEmitOnError.html)；[sourceMap](https://www.typescriptlang.org/tsconfig/sourceMap.html)；[skipLibCheck](https://www.typescriptlang.org/tsconfig/skipLibCheck.html)；[paths：不改写输出说明符](https://www.typescriptlang.org/tsconfig/paths.html)；[noUnusedLocals](https://www.typescriptlang.org/tsconfig/noUnusedLocals.html)；[noUnusedParameters](https://www.typescriptlang.org/tsconfig/noUnusedParameters.html)；[6.0：rootDir 与 types 默认值](https://www.typescriptlang.org/docs/handbook/release-notes/typescript-6-0.html)。 |
| Node.js 24.11.0 | [Packages：imports、exports、自引用](https://nodejs.org/download/release/v24.11.0/docs/api/packages.html)；[--enable-source-maps](https://nodejs.org/download/release/v24.11.0/docs/api/cli.html#--enable-source-maps)；[fs：copyFileSync](https://nodejs.org/download/release/v24.11.0/docs/api/fs.html)。 |
| npm | [npm ci](https://docs.npmjs.com/cli/v11/commands/npm-ci/)；[npm run](https://docs.npmjs.com/cli/v11/commands/npm-run/)。 |
| Microsoft Developer Blogs | [TypeScript 7.0：移除选项与编辑器支持](https://devblogs.microsoft.com/typescript/announcing-typescript-7-0/)。 |
